In [4]:
# Neural Identifier Training with Particle Filters - Van der Pol Oscillator
# With Different Equations Per Neuron in RHONN
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1) True nonlinear system (Van der Pol Oscillator)
# ============================================================
def plant_dynamics(x, u, mu=1.0):
    """
    Continuous dynamics for Van der Pol oscillator: x = [x1, x2]. 
    Returns x_dot.
    
    The Van der Pol equations:
    dx1/dt = x2
    dx2/dt = μ(1 - x1²)x2 - x1
    """
    x1, x2 = x
    
    # Van der Pol equations
    x1_dot = x2
    x2_dot = mu * (1 - x1**2) * x2 - x1
    
    return np.array([x1_dot, x2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# ============================================================
# 2) RHONN structure with different equations per neuron
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

# Define different feature functions for each neuron
def construct_z_vector_for_neuron(x_est, neuron_index):
    """
    Features for a 2-state Van der Pol oscillator, with different equations per neuron.
    
    Parameters:
    -----------
    x_est : array-like, shape (n_states,)
        Current state estimate
    neuron_index : int
        Index of the neuron (0 for x1, 1 for x2, etc.)
    
    Returns:
    --------
    z : array-like
        Feature vector for the specified neuron
    """
    s_x1 = sigmoidal(x_est[0])  # x1 (position)
    s_x2 = sigmoidal(x_est[1])  # x2 (velocity)
    
    if neuron_index == 0:  # Features for x1 equation (position)
        return np.array([
            s_x1,                            # Sigmoid of x1
            # s_x2,                            # Sigmoid of x2
            # s_x1 * s_x2,                     # Cross term
            s_x2**2,                         # Quadratic term for x2
            # s_x1**3,                         # Cubic term for x1 (Van der Pol nonlinearity)
            # x_est[0],                      # Direct linear term for x1 (optional)
            # 1.0                            # Bias term (optional)
        ])
    
    elif neuron_index == 1:  # Features for x2 equation (velocity)
        return np.array([
            # s_x1,                            # Sigmoid of x1
            s_x2,                            # Sigmoid of x2
            s_x1 * s_x2,                     # Cross term
            # (1 - s_x1**2) * s_x2,            # Van der Pol specific term
            s_x1**2 * s_x2,                  # Higher order nonlinearity
            # s_x2**3,                         # Cubic term for x2
            # x_est[1],                      # Direct linear term for x2 (optional)
            # 1.0                            # Bias term (optional)
        ])
    
    else:
        raise ValueError(f"Neuron index {neuron_index} not implemented")

def get_num_features_per_neuron(num_neurons=2):
    """
    Get the number of features for each neuron.
    
    Parameters:
    -----------
    num_neurons : int
        Number of neurons (states)
    
    Returns:
    --------
    list : Number of features for each neuron
    """
    # Manually set based on the feature functions above
    if num_neurons == 2:
        return [2, 3]  # 5 features for neuron 0, 6 features for neuron 1
    else:
        raise ValueError(f"Only {num_neurons} neurons implemented")

def RHONN_predict(x_state_for_z, w_neuron, neuron_index):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z_i( x(k) , u(k) )
    
    Parameters:
    -----------
    x_state_for_z : array-like
        State vector for building features
    w_neuron : array-like
        Weight vector for this neuron
    neuron_index : int
        Index of the neuron
    
    Returns:
    --------
    float : Predicted next state for this neuron
    """
    z_i = construct_z_vector_for_neuron(x_state_for_z, neuron_index)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch for neuron {neuron_index}: "
                         f"z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# ============================================================
# 3) EKF trainer over weights with different equations per neuron
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    Supports different equations per neuron.
    """
    def __init__(self, num_neurons, num_features_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_features_per_neuron = num_features_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_features_per_neuron[i]) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_features_per_neuron[i]) * P_init)
            self.Q.append(np.eye(num_features_per_neuron[i]) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x1 (position - measured output for Van der Pol oscillator)

        for i in range(self.num_neurons):
            # Get feature vector for this neuron
            z_i = construct_z_vector_for_neuron(x_state_for_z, i)
            H_i = z_i.reshape(-1, 1)  # column vector

            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_features_per_neuron[i]) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            # e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_features_per_neuron[i]) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_features_per_neuron[i]) * 1e-6

# ============================================================
# 4) Particle Filter trainer over weights with different equations per neuron
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    Supports different equations per neuron.
    """
    def __init__(self, num_neurons, num_features_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_features_per_neuron = num_features_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        # Initialize particles for each neuron
        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                # Use provided initial weights
                base_weight = initial_weights[i]
            else:
                # Generate random initial weights
                base_weight = np.random.randn(num_features_per_neuron[i]) * 0.1
            
            # Generate particles by adding noise to base weight
            particles_i = base_weight + np.random.randn(n_particles, num_features_per_neuron[i]) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]
        
        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.searchsorted(cdf, u0 + np.arange(N) / N)

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_features_per_neuron[i]) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            # Get feature vector for this neuron
            z_i = construct_z_vector_for_neuron(x_state_for_z, i)
            
            w_mat = self.particles[i]  # (N, num_features_for_neuron_i)
            x_pred_particles = w_mat @ z_i  # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights with different equations per neuron
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    Supports different equations per neuron.
    """
    def __init__(self, num_neurons, num_features_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_features_per_neuron = num_features_per_neuron
        self.eta = eta
        
        # UKF parameters - can be neuron-specific if needed
        self.alpha = alpha
        self.beta = beta
        self.kappa = kappa
        
        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_features_per_neuron[i]) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_features_per_neuron[i]) * P_init)
            self.Q.append(np.eye(num_features_per_neuron[i]) * Q_init)
            self.R.append(np.array([R_init]))
            
            # Set kappa for this neuron if not provided
            if kappa is None:
                self.kappa = 3 - num_features_per_neuron[i]

    def _generate_sigma_points(self, mean, covariance, neuron_index):
        """Generate sigma points for UKF for a specific neuron."""
        n = len(mean)
        
        # Calculate lambda for this neuron
        lambda_ = self.alpha**2 * (n + self.kappa) - n
        
        # Weights for mean and covariance computation
        Wm = np.zeros(2 * n + 1)
        Wc = np.zeros(2 * n + 1)
        
        Wm[0] = lambda_ / (n + lambda_)
        Wc[0] = lambda_ / (n + lambda_) + (1 - self.alpha**2 + self.beta)
        
        for i in range(1, 2 * n + 1):
            Wm[i] = 1.0 / (2 * (n + lambda_))
            Wc[i] = 1.0 / (2 * (n + lambda_))
        
        sigma_points = np.zeros((2 * n + 1, n))
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points, Wm, Wc

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)

        for i in range(self.num_neurons):
            # Get feature vector for this neuron
            z_i = construct_z_vector_for_neuron(x_state_for_z, i)
            
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points, Wm, Wc = self._generate_sigma_points(self.weights[i], self.P[i], i)
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            n = self.num_features_per_neuron[i]
            for j in range(2 * n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * n + 1)
            for j in range(2 * n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(n)
            for j in range(2 * n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(n) * 1e-6

# ============================================================
# 5) Parameter Optimization (Optional) - Updated for different equations per neuron
# ============================================================
def run_simulation_with_params(params, filter_type='UKF', n_steps=500, verbose=False):
    """
    Run a simulation with given parameters and return MSE.
    Updated for different equations per neuron.
    """
    dt = 0.01
    process_noise_type = 'gaussian'
    process_noise_std = 0.01
    
    # Initialize true system
    x_true = np.zeros((n_steps, 2))
    x_true[0] = [2.0, 0.0]
    u = 0.0
    
    # RHONN config with different equations per neuron
    num_neurons = 2
    num_features_per_neuron = get_num_features_per_neuron(num_neurons)
    
    # Fixed initial weights for each neuron
    np.random.seed(7517)
    common_initial_weights = []
    for i in range(num_neurons):
        # Different number of weights for each neuron
        common_initial_weights.append(np.random.uniform(-1.0, 1.0, num_features_per_neuron[i]))
    
    try:
        if filter_type == 'UKF':
            # Unpack parameters
            Q_init = 10 ** params[0]
            R_init = 10 ** params[1]
            P_init = 10 ** params[2]
            eta = params[3]
            alpha = 10 ** params[4]
            
            trainer = UKF_RHONN_Trainer(
                num_neurons, num_features_per_neuron,
                initial_weights=common_initial_weights,
                Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta,
                alpha=alpha, beta=2.0
            )
            x_hat = np.zeros((n_steps, 2))
            x_hat[0] = x_true[0]
            
        elif filter_type == 'PF':
            # Unpack parameters
            Q_std = 10 ** params[0]
            R_std = 10 ** params[1]
            n_particles_ratio = params[2]
            n_particles = int(100 * n_particles_ratio)
            
            trainer = PF_RHONN_Trainer(
                num_neurons, num_features_per_neuron,
                n_particles=n_particles,
                initial_weights=common_initial_weights,
                Q_std=Q_std, R_std=R_std, ess_threshold=n_particles / 2
            )
            
            # Initialize PF with common weights
            for i in range(trainer.num_neurons):
                trainer.particles[i] = np.tile(
                    common_initial_weights[i], (n_particles, 1)
                )
                trainer.weights_pf[i] = np.ones(n_particles) / n_particles
            
            x_hat = np.zeros((n_steps, 2))
            x_hat[0] = x_true[0]
            
        elif filter_type == 'EKF':
            # Unpack parameters
            Q_init = 10 ** params[0]
            R_init = 10 ** params[1]
            P_init = 10 ** params[2]
            eta = params[3]
            
            trainer = EKF_RHONN_Trainer(
                num_neurons, num_features_per_neuron,
                initial_weights=common_initial_weights,
                Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta
            )
            x_hat = np.zeros((n_steps, 2))
            x_hat[0] = x_true[0]
        
        else:
            raise ValueError(f"Unknown filter type: {filter_type}")
        
        # Run simulation
        for k in range(n_steps - 1):
            # True system step
            x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std)
            
            # Filter update
            trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat[k])
            
            # Prediction with neuron-specific equations
            x_state_for_z = np.copy(x_hat[k])
            x_state_for_z[0] = x_true[k][0]
            
            if filter_type == 'PF':
                weight_estimates = trainer.get_estimate()
                # Use neuron-specific prediction for each state
                x_hat[k+1, 0] = RHONN_predict(x_state_for_z, weight_estimates[0], neuron_index=0)
                x_hat[k+1, 1] = RHONN_predict(x_state_for_z, weight_estimates[1], neuron_index=1)
            else:
                # Use neuron-specific prediction for each state
                x_hat[k+1, 0] = RHONN_predict(x_state_for_z, trainer.weights[0], neuron_index=0)
                x_hat[k+1, 1] = RHONN_predict(x_state_for_z, trainer.weights[1], neuron_index=1)
        
        # Calculate MSE
        mse_x1 = np.mean((x_true[:, 0] - x_hat[:, 0])**2)
        mse_x2 = np.mean((x_true[:, 1] - x_hat[:, 1])**2)
        total_mse = mse_x1 + mse_x2
        
        if verbose:
            print(f"  MSE: {total_mse:.6e} | Params: {params}")
        
        return total_mse
    
    except Exception as e:
        if verbose:
            print(f"  Error with params {params}: {e}")
        return 1e10

def optimize_filter_parameters(filter_type='UKF', method='differential_evolution', 
                               n_steps=500, maxiter=50, verbose=True):
    """
    Optimize filter parameters using scipy optimization.
    Updated for different equations per neuron.
    """
    print(f"\n{'='*70}")
    print(f"🔧 OPTIMIZING {filter_type} PARAMETERS (Different equations per neuron)")
    print(f"{'='*70}")
    print(f"Method: {method}")
    print(f"Simulation length: {n_steps} steps")
    print(f"Max iterations: {maxiter}\n")
    
    # Define parameter bounds and initial guesses
    if filter_type == 'UKF':
        bounds = [(-6, -2), (-4, -1), (-1, 1), (0.1, 2.0), (-4, -1)]
        x0 = [-5, -2, 0, 1.0, -3]
        param_names = ['log10(Q_init)', 'log10(R_init)', 'log10(P_init)', 'eta', 'log10(alpha)']
        
    elif filter_type == 'PF':
        bounds = [(-2, 1), (-4, -2), (1, 10)]
        x0 = [0, -3, 8]
        param_names = ['log10(Q_std)', 'log10(R_std)', 'n_particles_ratio']
        
    elif filter_type == 'EKF':
        bounds = [(-6, -2), (-4, -1), (-1, 1), (0.1, 2.0)]
        x0 = [-3, -2, 0, 0.5]
        param_names = ['log10(Q_init)', 'log10(R_init)', 'log10(P_init)', 'eta']
    else:
        raise ValueError(f"Unknown filter type: {filter_type}")
    
    # Objective function
    objective = lambda params: run_simulation_with_params(params, filter_type, n_steps, verbose=False)
    
    # Optimize
    if method == 'differential_evolution':
        result = differential_evolution(
            objective, 
            bounds, 
            maxiter=maxiter,
            popsize=15,
            seed=42,
            atol=1e-6,
            tol=1e-6,
            workers=1,
            updating='deferred',
            disp=verbose
        )
    elif method == 'nelder-mead':
        result = minimize(
            objective,
            x0,
            method='Nelder-Mead',
            options={'maxiter': maxiter, 'disp': verbose, 'xatol': 1e-6, 'fatol': 1e-6}
        )
    else:
        raise ValueError(f"Unknown optimization method: {method}")
    
    # Extract and display results
    optimized_params = result.x
    final_mse = result.fun
    
    print(f"\n{'='*70}")
    print(f"✅ OPTIMIZATION COMPLETE")
    print(f"{'='*70}")
    print(f"Final MSE: {final_mse:.6e}")
    print(f"\nOptimized Parameters:")
    
    # Display optimized parameters
    for name, value in zip(param_names, optimized_params):
        print(f"  {name}: {value:.6f}")
    
    # Return dictionary with optimized parameters
    if filter_type == 'UKF':
        return {
            'Q_init': 10 ** optimized_params[0],
            'R_init': 10 ** optimized_params[1],
            'P_init': 10 ** optimized_params[2],
            'eta': optimized_params[3],
            'alpha': 10 ** optimized_params[4],
            'beta': 2.0,
            'mse': final_mse
        }
    elif filter_type == 'PF':
        return {
            'Q_std': 10 ** optimized_params[0],
            'R_std': 10 ** optimized_params[1],
            'n_particles': int(100 * optimized_params[2]),
            'mse': final_mse
        }
    elif filter_type == 'EKF':
        return {
            'Q_init': 10 ** optimized_params[0],
            'R_init': 10 ** optimized_params[1],
            'P_init': 10 ** optimized_params[2],
            'eta': optimized_params[3],
            'mse': final_mse
        }

# ============================================================
# 6) Simulation with different equations per neuron
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'gaussian'
    process_noise_std = 0.01
    measurement_noise_std = 0.05

    # --- True system init ---
    x_true = np.zeros((n_steps, 2))
    x_true[0] = [2.0, 0.0]  # Initial conditions for Van der Pol oscillator
    u = 0.0

    # --- RHONN config with different equations per neuron ---
    num_neurons = 2
    num_features_per_neuron = get_num_features_per_neuron(num_neurons)
    
    print(f"Number of features per neuron: {num_features_per_neuron}")
    print(f"Total features: {sum(num_features_per_neuron)}")

    # --- Common initial weights for each neuron ---
    np.random.seed(7517)
    common_initial_weights = []
    for i in range(num_neurons):
        common_initial_weights.append(np.random.uniform(-1.0, 1.0, num_features_per_neuron[i]))
    
    print("\nCommon Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i} (x{i+1}): {w} (shape: {w.shape})")

    # --- UKF with different equations per neuron ---
    ukf_trainer = UKF_RHONN_Trainer(
        num_neurons, num_features_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1.0e-2, R_init=1.0e-4, P_init=2.121261, eta=1.0436,
        alpha=2.897616e-4, beta=2.0
    )
    x_hat_ukf = np.zeros((n_steps, 2))
    x_hat_ukf[0] = x_true[0]
    # x_hat_ukf[0] = x_true[0] + [0.5, 0.5]

    # --- PF with different equations per neuron ---
    n_particles = 980
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_features_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=1.0, R_std=0.002, ess_threshold=n_particles * 0.5
    )

    # Force identical particle initialization
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 2))
    x_hat_pf[0] = x_true[0]
    # x_hat_pf[0] = x_true[0] + [0.5, 0.5]

    print("\nStarting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std) + np.random.randn(num_neurons) * measurement_noise_std

        # ---- 2b) UKF update with different equations per neuron ----
        ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ukf[k])

        x_state_for_z_ukf = np.copy(x_hat_ukf[k])
        x_state_for_z_ukf[0] = x_true[k][0]  # series-parallel uses measured x1 at k
        
        # Use neuron-specific prediction
        x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], neuron_index=0)
        x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], neuron_index=1)

        # ---- 3) PF update with different equations per neuron ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0]  # series-parallel uses measured x1 at k
        
        # Use neuron-specific prediction
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0], neuron_index=0)
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1], neuron_index=1)

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

    # ============================================================
    # 7) Resultados y gráficas para Oscilador de Van der Pol
    # ============================================================

    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }

    # Cálculo de MSE
    mse_x1_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
    mse_x2_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
    mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
    mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)

    mse_total_ukf = mse_x1_ukf + mse_x2_ukf
    mse_total_pf = mse_x1_pf + mse_x2_pf

    print("="*70)
    print(f"🏆 MEJOR FILTRO: ", end="")
    mse_dict = {'UKF': mse_total_ukf, 'PF': mse_total_pf}
    best_filter = min(mse_dict, key=mse_dict.get)
    print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
    print("="*70)

    print(f"\nPesos Finales UKF-RHONN (diferentes ecuaciones por neurona):")
    for i in range(2):
        print(f"  Neurona {i+1} (x{i+1}): {ukf_trainer.weights[i]} (shape: {ukf_trainer.weights[i].shape})")

    print(f"\nEstimación de Pesos PF-RHONN (diferentes ecuaciones por neurona):")
    pf_estimates = pf_trainer.get_estimate()
    for i in range(2):
        print(f"  Neurona {i+1} (x{i+1}): {pf_estimates[i]} (shape: {pf_estimates[i].shape})")

    print("\n--- Comparación de Desempeño (MSE) - Oscilador de Van der Pol ---")
    print(f"UKF MSE x₁ (posición):  {mse_x1_ukf:.6f}")
    print(f"UKF MSE x₂ (velocidad): {mse_x2_ukf:.6f}")
    print(f"PF  MSE x₁ (posición):  {mse_x1_pf:.6f}")
    print(f"PF  MSE x₂ (velocidad): {mse_x2_pf:.6f}")

    # Gráficas individuales por estado
    states_info = [
        {'idx': 0, 'var': 'x₁', 'desc': 'Posición', 'y_label': 'Posición x₁'},
        {'idx': 1, 'var': 'x₂', 'desc': 'Velocidad', 'y_label': 'Velocidad x₂'}
    ]

    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        # Estado real (línea negra gruesa)
        fig.add_trace(go.Scatter(
            x=t_history, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        # Estimación UKF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))
        
        # Estimación PF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        # Set legend position: top right for x1, bottom right for x2
        legend_y = 0.98 if i == 0 else 0.02
        legend_yanchor = 'top' if i == 0 else 'bottom'
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Oscilador de Van der Pol',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.98,
                y=legend_y,
                xanchor='right',
                yanchor=legend_yanchor,
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )
        
        fig.show()

    # Gráfica de errores combinada
    error_x1_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
    error_x2_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
    error_x1_pf = x_true[:, 0] - x_hat_pf[:, 0]
    error_x2_pf = x_true[:, 1] - x_hat_pf[:, 1]

    fig_err = go.Figure()

    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_x1_ukf,
        mode='lines',
        name=f'UKF Error x₁ (MSE={mse_x1_ukf:.2e})',
        line=dict(color='#2ca02c', width=1.5),
        opacity=0.8
    ))

    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_x1_pf,
        mode='lines',
        name=f'PF Error x₁ (MSE={mse_x1_pf:.2e})',
        line=dict(color='#d62728', width=1.5),
        opacity=0.8
    ))

    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_x2_ukf,
        mode='lines',
        name=f'UKF Error x₂ (MSE={mse_x2_ukf:.2e})',
        line=dict(color='#2ca02c', width=1.5, dash='dot'),
        opacity=0.8
    ))

    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_x2_pf,
        mode='lines',
        name=f'PF Error x₂ (MSE={mse_x2_pf:.2e})',
        line=dict(color='#d62728', width=1.5, dash='dot'),
        opacity=0.8
    ))

    fig_err.update_layout(
        title={
            'text': 'Errores de Estimación - Oscilador de Van der Pol',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title='Error de Estimación',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_err.show()

    # Espacio de fase (Ciclo Límite)
    fig_phase = go.Figure()

    fig_phase.add_trace(go.Scatter(
        x=x_true[:, 0], y=x_true[:, 1],
        mode='lines',
        name='Ciclo Límite Real',
        line=dict(color='#000000', width=3)
    ))

    fig_phase.add_trace(go.Scatter(
        x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1],
        mode='lines',
        name='Estimación UKF-RHONN',
        line=dict(color='#2ca02c', width=2, dash='dot')
    ))

    fig_phase.add_trace(go.Scatter(
        x=x_hat_pf[:, 0], y=x_hat_pf[:, 1],
        mode='lines',
        name='Estimación PF-RHONN',
        line=dict(color='#d62728', width=2, dash='dashdot')
    ))

    fig_phase.update_layout(
        title={
            'text': 'Espacio de Fases - Oscilador de Van der Pol',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Posición x₁',
        yaxis_title='Velocidad x₂',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.02,
            xanchor='right',
            yanchor='bottom',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_width'],  # Aspecto cuadrado
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_phase.show()

    # Gráfica de barras comparando MSE
    fig_mse = go.Figure()

    filters = ['UKF-RHONN', 'PF-RHONN']

    fig_mse.add_trace(go.Bar(
        name='Posición x₁',
        x=filters,
        y=[mse_x1_ukf, mse_x1_pf],
        marker_color='#636EFA',
        text=[f'{mse_x1_ukf:.2e}', f'{mse_x1_pf:.2e}'],
        textposition='outside'
    ))

    fig_mse.add_trace(go.Bar(
        name='Velocidad x₂',
        x=filters,
        y=[mse_x2_ukf, mse_x2_pf],
        marker_color='#EF553B',
        text=[f'{mse_x2_ukf:.2e}', f'{mse_x2_pf:.2e}'],
        textposition='outside'
    ))

    fig_mse.update_layout(
        title={
            'text': 'Comparación de Error Cuadrático Medio (MSE)',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Error Cuadrático Medio (MSE)',
        yaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_mse.show()

    # ============================================================
    # 8) RMSE Calculation and Plots
    # ============================================================

    # Calculate RMSE for each filter and state
    rmse_x1_ukf = np.sqrt(mse_x1_ukf)
    rmse_x2_ukf = np.sqrt(mse_x2_ukf)
    rmse_x1_pf = np.sqrt(mse_x1_pf)
    rmse_x2_pf = np.sqrt(mse_x2_pf)

    # Total RMSE (combined metric)
    rmse_total_ukf = np.sqrt(mse_total_ukf)
    rmse_total_pf = np.sqrt(mse_total_pf)

    # Print RMSE comparison
    print("\n" + "="*70)
    print("📊 ROOT MEAN SQUARE ERROR (RMSE) COMPARISON")
    print("="*70)
    print("\nUKF-RHONN:")
    print(f"  RMSE x₁ (posición):  {rmse_x1_ukf:.6f}")
    print(f"  RMSE x₂ (velocidad): {rmse_x2_ukf:.6f}")
    print(f"  RMSE Total:          {rmse_total_ukf:.6f}")

    print("\nPF-RHONN:")
    print(f"  RMSE x₁ (posición):  {rmse_x1_pf:.6f}")
    print(f"  RMSE x₂ (velocidad): {rmse_x2_pf:.6f}")
    print(f"  RMSE Total:          {rmse_total_pf:.6f}")

    print("\n" + "="*70)
    print(f"🏆 MEJOR FILTRO (RMSE): ", end="")
    rmse_dict = {'UKF': rmse_total_ukf, 'PF': rmse_total_pf}
    best_filter_rmse = min(rmse_dict, key=rmse_dict.get)
    print(f"{best_filter_rmse} (RMSE total: {rmse_dict[best_filter_rmse]:.6f})")
    print("="*70)

    # Bar chart comparing RMSE
    fig_rmse = go.Figure()

    filters = ['UKF-RHONN', 'PF-RHONN']

    fig_rmse.add_trace(go.Bar(
        name='Posición x₁',
        x=filters,
        y=[rmse_x1_ukf, rmse_x1_pf],
        marker_color='#636EFA',
        text=[f'{rmse_x1_ukf:.4f}', f'{rmse_x1_pf:.4f}'],
        textposition='outside'
    ))

    fig_rmse.add_trace(go.Bar(
        name='Velocidad x₂',
        x=filters,
        y=[rmse_x2_ukf, rmse_x2_pf],
        marker_color='#EF553B',
        text=[f'{rmse_x2_ukf:.4f}', f'{rmse_x2_pf:.4f}'],
        textposition='outside'
    ))

    fig_rmse.add_trace(go.Bar(
        name='RMSE Total',
        x=filters,
        y=[rmse_total_ukf, rmse_total_pf],
        marker_color='#00CC96',
        text=[f'{rmse_total_ukf:.4f}', f'{rmse_total_pf:.4f}'],
        textposition='outside'
    ))

    fig_rmse.update_layout(
        title={
            'text': 'Comparación de Error Cuadrático Medio Raíz (RMSE)',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Error Cuadrático Medio Raíz (RMSE)',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_rmse.show()

    # Time-varying RMSE (cumulative moving window)
    window_size = 50  # 50 time steps moving window

    rmse_time_x1_ukf = np.zeros(n_steps - window_size)
    rmse_time_x2_ukf = np.zeros(n_steps - window_size)
    rmse_time_x1_pf = np.zeros(n_steps - window_size)
    rmse_time_x2_pf = np.zeros(n_steps - window_size)

    for i in range(n_steps - window_size):
        rmse_time_x1_ukf[i] = np.sqrt(np.mean((x_true[i:i+window_size, 0] - x_hat_ukf[i:i+window_size, 0])**2))
        rmse_time_x2_ukf[i] = np.sqrt(np.mean((x_true[i:i+window_size, 1] - x_hat_ukf[i:i+window_size, 1])**2))
        rmse_time_x1_pf[i] = np.sqrt(np.mean((x_true[i:i+window_size, 0] - x_hat_pf[i:i+window_size, 0])**2))
        rmse_time_x2_pf[i] = np.sqrt(np.mean((x_true[i:i+window_size, 1] - x_hat_pf[i:i+window_size, 1])**2))

    t_rmse = t_history[window_size:]

    # Plot time-varying RMSE
    fig_rmse_time = go.Figure()

    fig_rmse_time.add_trace(go.Scatter(
        x=t_rmse, y=rmse_time_x1_ukf,
        mode='lines',
        name='UKF RMSE x₁',
        line=dict(color='#2ca02c', width=2)
    ))

    fig_rmse_time.add_trace(go.Scatter(
        x=t_rmse, y=rmse_time_x1_pf,
        mode='lines',
        name='PF RMSE x₁',
        line=dict(color='#d62728', width=2)
    ))

    fig_rmse_time.add_trace(go.Scatter(
        x=t_rmse, y=rmse_time_x2_ukf,
        mode='lines',
        name='UKF RMSE x₂',
        line=dict(color='#2ca02c', width=2, dash='dot')
    ))

    fig_rmse_time.add_trace(go.Scatter(
        x=t_rmse, y=rmse_time_x2_pf,
        mode='lines',
        name='PF RMSE x₂',
        line=dict(color='#d62728', width=2, dash='dot')
    ))

    fig_rmse_time.update_layout(
        title={
            'text': f'RMSE en Ventana Móvil (ventana={window_size} pasos)',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title='RMSE en Ventana Móvil',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_rmse_time.show()

    print("\nSimulation and plotting completed successfully!")



Number of features per neuron: [2, 3]
Total features: 5

Common Initial Weights:
  Neuron 0 (x1): [ 0.18387164 -0.7100519 ] (shape: (2,))
  Neuron 1 (x2): [ 0.31646359 -0.55559831  0.98591761] (shape: (3,))

Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 80.0%
Simulation progress: 90.0%
Simulation finished.
🏆 MEJOR FILTRO: PF (MSE total: 0.000002)

Pesos Finales UKF-RHONN (diferentes ecuaciones por neurona):
  Neurona 1 (x1): [-1.75480638 10.64585894] (shape: (2,))
  Neurona 2 (x2): [ -0.58856903 -16.78643412   5.24433211] (shape: (3,))

Estimación de Pesos PF-RHONN (diferentes ecuaciones por neurona):
  Neurona 1 (x1): [  1.57004547 -24.75336475] (shape: (2,))
  Neurona 2 (x2): [-8.75885442  4.26188902  0.85811327] (shape: (3,))

--- Comparación de Desempeño (MSE) - Oscilador 


📊 ROOT MEAN SQUARE ERROR (RMSE) COMPARISON

UKF-RHONN:
  RMSE x₁ (posición):  0.005898
  RMSE x₂ (velocidad): 0.006429
  RMSE Total:          0.008724

PF-RHONN:
  RMSE x₁ (posición):  0.001175
  RMSE x₂ (velocidad): 0.000984
  RMSE Total:          0.001533

🏆 MEJOR FILTRO (RMSE): PF (RMSE total: 0.001533)



Simulation and plotting completed successfully!


In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1) True nonlinear system (Double Pendulum)
# ============================================================
def double_pendulum_dynamics(state, t=0, m1=1.0, m2=1.0, L1=1.0, L2=1.0, g=9.81):
    """
    Continuous dynamics for double pendulum.
    State vector: [θ1, θ2, ω1, ω2] where:
        θ1, θ2: angles of the two pendulums (radians)
        ω1, ω2: angular velocities (radians/s)

    Returns state derivatives.
    """
    θ1, θ2, ω1, ω2 = state

    # Common terms
    Δθ = θ2 - θ1
    sin_Δθ = np.sin(Δθ)
    cos_Δθ = np.cos(Δθ)
    sin_θ1 = np.sin(θ1)
    sin_θ2 = np.sin(θ2)

    # Denominator
    denom = m2 * L2 * cos_Δθ**2 - (m1 + m2) * L1

    # Avoid division by zero
    if np.abs(denom) < 1e-6:
        denom = 1e-6 if denom >= 0 else -1e-6

    # Equations of motion for ω1_dot and ω2_dot
    ω1_dot = (m2 * L2 * ω2**2 * sin_Δθ * cos_Δθ +
              m2 * g * sin_θ2 * cos_Δθ +
              m2 * L2 * ω1**2 * sin_Δθ -
              (m1 + m2) * g * sin_θ1) / (L1 * denom)

    ω2_dot = (-m2 * L2 * ω2**2 * sin_Δθ * cos_Δθ +
              (m1 + m2) * (g * sin_θ1 * cos_Δθ -
                           L1 * ω1**2 * sin_Δθ -
                           g * sin_θ2)) / (L2 * denom)

    # Derivatives
    θ1_dot = ω1
    θ2_dot = ω2

    return np.array([θ1_dot, θ2_dot, ω1_dot, ω2_dot])

def plant(x_k, u_k=0.0, dt=0.01, process_noise_type='gaussian', process_noise_std=1e-3):
    """
    One Euler step of the discrete double pendulum with process noise.
    """
    x_dot = double_pendulum_dynamics(x_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# ============================================================
# 2) RHONN structure with different equations per neuron for double pendulum
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -800, 800)
    return 1.0 / (1.0 + np.exp(-beta * z))

# Define different feature functions for each neuron (4 neurons for 4 states)
def construct_z_vector_for_neuron(x_est, neuron_index):
    """
    Features for a 4-state double pendulum, with different equations per neuron.

    Parameters:
    -----------
    x_est : array-like, shape (4,)
        Current state estimate [θ1, θ2, ω1, ω2]
    neuron_index : int
        Index of the neuron (0: θ1, 1: θ2, 2: ω1, 3: ω2)

    Returns:
    --------
    z : array-like
        Feature vector for the specified neuron
    """
    # Extract states
    θ1, θ2, ω1, ω2 = x_est

    # Compute sigmoid of states
    s_θ1 = sigmoidal(θ1)
    s_θ2 = sigmoidal(θ2)
    s_ω1 = sigmoidal(ω1)
    s_ω2 = sigmoidal(ω2)

    # Common terms
    sin_θ1 = np.sin(θ1)
    sin_θ2 = np.sin(θ2)
    cos_θ1 = np.cos(θ1)
    cos_θ2 = np.cos(θ2)
    Δθ = θ2 - θ1
    sin_Δθ = np.sin(Δθ)
    cos_Δθ = np.cos(Δθ)

    if neuron_index == 0:  # Features for θ1 (angle of first pendulum)
        return np.array([
            s_θ1,                            # Sigmoid of θ1
            # s_ω1,                            # Sigmoid of ω1
            s_θ1 * s_ω1,                     # Cross term
            # sin_θ1,                          # sin(θ1) - important for pendulum
            s_θ1**2,                         # Quadratic term
            # s_θ1 * s_θ2,                     # Interaction with θ2
            # 1.0                              # Bias term
        ])

    elif neuron_index == 1:  # Features for θ2 (angle of second pendulum)
        return np.array([
            s_θ2,                            # Sigmoid of θ2
            # s_ω2,                            # Sigmoid of ω2
            s_θ2 * s_ω2,                     # Cross term
            # sin_θ2,                          # sin(θ2)
            s_θ2**3,                         # Quadratic term
            # s_θ1 * s_θ2,                     # Interaction with θ1
            # sin_Δθ,                          # sin(θ2 - θ1) - coupling term
            # 1.0                              # Bias term
        ])

    elif neuron_index == 2:  # Features for ω1 (angular velocity of first pendulum)
        return np.array([
            s_ω1,                            # Sigmoid of ω1
            s_θ1,                            # Sigmoid of θ1
            # s_θ2,                            # Sigmoid of θ2
            s_ω2,                            # Sigmoid of ω2
            # s_ω1**3,                         # Quadratic in ω1
            s_θ1 * s_ω1,                     # θ1-ω1 interaction
            # sin_θ1,                          # sin(θ1) - gravity term
            # sin_Δθ * cos_Δθ,                 # sin(Δθ)cos(Δθ) - coupling term
            # s_ω2**2 * sin_Δθ,                # ω2² sin(Δθ) - centrifugal term
            # 1.0                              # Bias term
        ])

    elif neuron_index == 3:  # Features for ω2 (angular velocity of second pendulum)
        return np.array([
            s_ω2,                            # Sigmoid of ω2
            # s_θ1,                            # Sigmoid of θ1
            s_θ2,                            # Sigmoid of θ2
            # s_ω1,                            # Sigmoid of ω1
            s_ω2**2,                         # Quadratic in ω2
            s_θ2 * s_ω2,                     # θ2-ω2 interaction
            # sin_θ2,                          # sin(θ2) - gravity term
            # sin_Δθ * cos_Δθ,                 # sin(Δθ)cos(Δθ) - coupling term
            # s_ω1**2 * sin_Δθ,                # ω1² sin(Δθ) - centrifugal term
            # cos_Δθ * sin_θ1,                 # cos(Δθ) sin(θ1) - complex coupling
            # 1.0                              # Bias term
        ])

    else:
        raise ValueError(f"Neuron index {neuron_index} not implemented for 4-state system")

def get_num_features_per_neuron(num_neurons=4):
    """
    Get the number of features for each neuron.

    Parameters:
    -----------
    num_neurons : int
        Number of neurons (states)

    Returns:
    --------
    list : Number of features for each neuron
    """
    # Based on the feature functions above for double pendulum
    if num_neurons == 4:
        return [3,3,4,4]  # Features per neuron: θ1, θ2, ω1, ω2 (including bias)
    else:
        # For testing with different numbers of neurons
        # Calculate dynamically using a dummy state
        dummy_state = np.zeros(num_neurons)
        num_features = []
        for i in range(num_neurons):
            try:
                z = construct_z_vector_for_neuron(dummy_state, i)
                num_features.append(len(z))
            except ValueError:
                raise ValueError(f"Neuron index {i} not implemented")
        return num_features

def RHONN_predict(x_state_for_z, w_neuron, neuron_index):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z_i( x(k) , u(k) )

    Parameters:
    -----------
    x_state_for_z : array-like
        State vector for building features
    w_neuron : array-like
        Weight vector for this neuron
    neuron_index : int
        Index of the neuron

    Returns:
    --------
    float : Predicted next state for this neuron
    """
    z_i = construct_z_vector_for_neuron(x_state_for_z, neuron_index)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch for neuron {neuron_index}: "
                         f"z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# ============================================================
# 3) EKF trainer over weights with different equations per neuron
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    Supports different equations per neuron.
    """
    def __init__(self, num_neurons, num_features_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_features_per_neuron = num_features_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_features_per_neuron[i]) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_features_per_neuron[i]) * P_init)
            self.Q.append(np.eye(num_features_per_neuron[i]) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        # For double pendulum, all states are typically measured
        # But we can choose which ones are actually measured in practice
        # Here we assume all states are measured
        x_state_for_z[:] = chi_k[:]

        for i in range(self.num_neurons):
            # Get feature vector for this neuron
            z_i = construct_z_vector_for_neuron(x_state_for_z, i)
            H_i = z_i.reshape(-1, 1)  # column vector

            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]

            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_features_per_neuron[i]) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i

            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_features_per_neuron[i]) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]

            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_features_per_neuron[i]) * 1e-6

# ============================================================
# 4) Particle Filter trainer over weights with different equations per neuron
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    Supports different equations per neuron.
    """
    def __init__(self, num_neurons, num_features_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_features_per_neuron = num_features_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        # Initialize particles for each neuron
        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                # Use provided initial weights
                base_weight = initial_weights[i]
            else:
                # Generate random initial weights
                base_weight = np.random.randn(num_features_per_neuron[i]) * 0.1

            # Generate particles by adding noise to base weight
            particles_i = base_weight + np.random.randn(n_particles, num_features_per_neuron[i]) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.searchsorted(cdf, u0 + np.arange(N) / N)

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        # Assume all states are measured
        x_state_for_z[:] = chi_k[:]

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_features_per_neuron[i]) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            # Get feature vector for this neuron
            z_i = construct_z_vector_for_neuron(x_state_for_z, i)

            w_mat = self.particles[i]  # (N, num_features_for_neuron_i)
            x_pred_particles = w_mat @ z_i  # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights with different equations per neuron
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    Supports different equations per neuron.
    """
    def __init__(self, num_neurons, num_features_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0,
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_features_per_neuron = num_features_per_neuron
        self.eta = eta

        # UKF parameters - can be neuron-specific if needed
        self.alpha = alpha
        self.beta = beta
        self._initial_kappa = kappa # Store the initial kappa value

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_features_per_neuron[i]) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_features_per_neuron[i]) * P_init)
            self.Q.append(np.eye(num_features_per_neuron[i]) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance, neuron_index):
        """Generate sigma points for UKF for a specific neuron."""
        n = len(mean)

        # Determine kappa for the current neuron's dimension
        if self._initial_kappa is None:
            # If no global kappa was set, use the default 3 - n for this specific neuron's n
            current_kappa = 3 - n
        else:
            # Otherwise, use the global kappa provided
            current_kappa = self._initial_kappa

        # Calculate lambda for this neuron
        lambda_ = self.alpha**2 * (n + current_kappa) - n

        # Weights for mean and covariance computation
        Wm = np.zeros(2 * n + 1)
        Wc = np.zeros(2 * n + 1)

        Wm[0] = lambda_ / (n + lambda_)
        Wc[0] = lambda_ / (n + lambda_) + (1 - self.alpha**2 + self.beta)

        for i in range(1, 2 * n + 1):
            Wm[i] = 1.0 / (2 * (n + lambda_))
            Wc[i] = 1.0 / (2 * (n + lambda_))

        sigma_points = np.zeros((2 * n + 1, n))
        sigma_points[0] = mean

        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + lambda_)

        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]

        return sigma_points, Wm, Wc

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        # Assume all states are measured
        x_state_for_z[:] = chi_k[:]

        for i in range(self.num_neurons):
            n = self.num_features_per_neuron[i]

            # --- Safety check: if P or weights have gone bad, reset them ---
            if np.any(np.isnan(self.P[i])) or np.any(np.isinf(self.P[i])):
                warnings.warn(f"UKF P[{i}] has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0 # Reset P to a reasonable value
                self.weights[i] = np.random.randn(n) * 0.1 # Reset weights
                continue # Skip this update step for the current neuron

            if np.any(np.isnan(self.weights[i])) or np.any(np.isinf(self.weights[i])):
                warnings.warn(f"UKF weights[{i}] has NaNs/Infs. Resetting weights.")
                self.weights[i] = np.random.randn(n) * 0.1 # Reset weights
                self.P[i] = np.eye(n) * 1.0 # Also reset P
                continue # Skip this update step for the current neuron

            # Get feature vector for this neuron
            z_i = construct_z_vector_for_neuron(x_state_for_z, i)
            # Ensure z_i is also clean
            if np.any(np.isnan(z_i)) or np.any(np.isinf(z_i)):
                warnings.warn(f"Feature vector z_i for neuron {i} has NaNs/Infs. Skipping update for this neuron.")
                continue

            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points, Wm, Wc = self._generate_sigma_points(self.weights[i], self.P[i], i)

            # Check sigma points for NaNs/Infs
            if np.any(np.isnan(sigma_points)) or np.any(np.isinf(sigma_points)):
                warnings.warn(f"Sigma points for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()

            # Predict mean and covariance
            predicted_mean = np.sum(Wm[:, np.newaxis] * predicted_sigma_points, axis=0)

            # Check predicted_mean for NaNs/Infs
            if np.any(np.isnan(predicted_mean)) or np.any(np.isinf(predicted_mean)):
                warnings.warn(f"Predicted mean for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            predicted_cov = self.Q[i].copy()
            for j in range(2 * n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += Wc[j] * np.outer(diff, diff)

            # Check predicted_cov for NaNs/Infs
            if np.any(np.isnan(predicted_cov)) or np.any(np.isinf(predicted_cov)):
                warnings.warn(f"Predicted covariance for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * n + 1)
            for j in range(2 * n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)

            # Check measurement_sigma_points for NaNs/Infs
            if np.any(np.isnan(measurement_sigma_points)) or np.any(np.isinf(measurement_sigma_points)):
                warnings.warn(f"Measurement sigma points for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            # Predicted measurement mean
            predicted_measurement = np.sum(Wm * measurement_sigma_points)

            # Check predicted_measurement for NaNs/Infs
            if np.isnan(predicted_measurement) or np.isinf(predicted_measurement):
                warnings.warn(f"Predicted measurement for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += Wc[j] * diff_meas**2

            # Check innovation_cov for NaNs/Infs
            if np.isnan(innovation_cov) or np.isinf(innovation_cov):
                warnings.warn(f"Innovation covariance for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            # Cross-covariance
            cross_cov = np.zeros(n)
            for j in range(2 * n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += Wc[j] * diff_state * diff_meas

            # Check cross_cov for NaNs/Infs
            if np.any(np.isnan(cross_cov)) or np.any(np.isinf(cross_cov)):
                warnings.warn(f"Cross-covariance for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov

            # Check Kalman gain K for NaNs/Infs
            if np.any(np.isnan(K)) or np.any(np.isinf(K)):
                warnings.warn(f"Kalman Gain K for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            # Innovation
            innovation = chi_kp1[i] - predicted_measurement

            # Check innovation for NaNs/Infs
            if np.isnan(innovation) or np.isinf(innovation):
                warnings.warn(f"Innovation for neuron {i} has NaNs/Infs. Resetting P and weights.")
                self.P[i] = np.eye(n) * 1.0
                self.weights[i] = np.random.randn(n) * 0.1
                continue

            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation

            # Check updated weights for NaNs/Infs
            if np.any(np.isnan(self.weights[i])) or np.any(np.isinf(self.weights[i])):
                warnings.warn(f"Updated weights[{i}] has NaNs/Infs. Resetting weights.")
                self.weights[i] = np.random.randn(n) * 0.1
                self.P[i] = np.eye(n) * 1.0
                continue

            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov

            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)

            # Final check before eigenvals
            if np.any(np.isnan(self.P[i])) or np.any(np.isinf(self.P[i])):
                warnings.warn(f"Final P[{i}] after update has NaNs/Infs. Resetting P.")
                self.P[i] = np.eye(n) * 1.0
                continue

            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(n) * 1e-6

# ============================================================
# 5) Simulation for Double Pendulum with different equations per neuron
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000  # Reduced for faster execution
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'gaussian'
    process_noise_std = 0.01
    measurement_noise_std = 0.02  # Slightly higher noise for chaotic system

    # --- True system init ---
    x_true = np.zeros((n_steps, 4))
    # Initial conditions: both pendulums at 90 degrees with small velocities
    x_true[0] = [np.pi/2, np.pi/2, 0.1, 0.1]
    u = 0.0
    
    # --- Noisy measurements ---
    chi = np.zeros((n_steps, 4))  # Noisy measurements
    chi[0] = x_true[0] + np.random.randn(4) * measurement_noise_std

    # --- RHONN config with different equations per neuron ---
    num_neurons = 4
    num_features_per_neuron = get_num_features_per_neuron(num_neurons)

    print(f"Double Pendulum System Identification")
    print(f"=====================================")
    print(f"Number of features per neuron: {num_features_per_neuron}")
    print(f"Total features: {sum(num_features_per_neuron)}")
    print(f"States: [θ1, θ2, ω1, ω2]")

    # --- Common initial weights for each neuron ---
    np.random.seed(7517)
    common_initial_weights = []
    for i in range(num_neurons):
        common_initial_weights.append(np.random.uniform(-1.0, 1.0, num_features_per_neuron[i]))

    print("\nCommon Initial Weights (different dimensions per neuron):")
    state_names = ['θ1 (angle 1)', 'θ2 (angle 2)', 'ω1 (velocity 1)', 'ω2 (velocity 2)']
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i} ({state_names[i]}): shape = {w.shape}")

    # --- UKF with different equations per neuron ---
    # Using pre-tuned parameters for double pendulum
    ukf_trainer = UKF_RHONN_Trainer(
        num_neurons, num_features_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1.0e-3, R_init=1.0e-5, P_init=0.1, eta=0.8, # Changed R_init, P_init
        alpha=0.1, beta=2.0 # Changed alpha
    )
    x_hat_ukf = np.zeros((n_steps, 4))
    x_hat_ukf[0] = x_true[0]

    # --- PF with different equations per neuron ---
    n_particles = 1000  # Reduced for computational efficiency
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_features_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.5, R_std=0.01, ess_threshold=n_particles / 2
    )

    # Force identical particle initialization
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 4))
    x_hat_pf[0] = x_true[0]

    print("\nStarting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std)
        
        # Create noisy measurement (separate from true state)
        chi[k+1] = x_true[k+1] + np.random.randn(4) * measurement_noise_std

        # ---- 2b) UKF update with different equations per neuron ----
        ukf_trainer.update(chi_kp1=chi[k+1], chi_k=chi[k], x_hat_previous=x_hat_ukf[k])

        x_state_for_z_ukf = np.copy(x_hat_ukf[k])
        # Series-parallel: use measured states for feature construction
        x_state_for_z_ukf[:] = chi[k][:]

        # Use neuron-specific prediction for all 4 states
        for i in range(num_neurons):
            x_hat_ukf[k+1, i] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[i], neuron_index=i)

        # ---- 3) PF update with different equations per neuron ----
        pf_trainer.update(chi_kp1=chi[k+1], chi_k=chi[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[:] = chi[k][:]

        # Use neuron-specific prediction for all 4 states
        for i in range(num_neurons):
            x_hat_pf[k+1, i] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[i], neuron_index=i)

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

    # ============================================================
    # 6) Results and Visualization for Double Pendulum
    # ============================================================

    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }

    # Calculate MSE for each state
    mse_states_ukf = []
    mse_states_pf = []

    for i in range(num_neurons):
        mse_ukf = np.mean((x_true[:, i] - x_hat_ukf[:, i])**2)
        mse_pf = np.mean((x_true[:, i] - x_hat_pf[:, i])**2)
        mse_states_ukf.append(mse_ukf)
        mse_states_pf.append(mse_pf)

    mse_total_ukf = np.sum(mse_states_ukf)
    mse_total_pf = np.sum(mse_states_pf)

    print("\n" + "="*70)
    print(f"🏆 BEST FILTER: ", end="")
    mse_dict = {'UKF': mse_total_ukf, 'PF': mse_total_pf}
    best_filter = min(mse_dict, key=mse_dict.get)
    print(f"{best_filter} (Total MSE: {mse_dict[best_filter]:.6f})")
    print("="*70)

    print(f"\nFinal Weights UKF-RHONN (different equations per neuron):")
    for i in range(num_neurons):
        print(f"  Neuron {i} ({state_names[i]}): shape = {ukf_trainer.weights[i].shape}")

    print(f"\nWeight Estimates PF-RHONN (different equations per neuron):")
    pf_estimates = pf_trainer.get_estimate()
    for i in range(num_neurons):
        print(f"  Neuron {i} ({state_names[i]}): shape = {pf_estimates[i].shape}")

    print("\n-- Performance Comparison (MSE) - Double Pendulum ---")
    for i in range(num_neurons):
        print(f"UKF MSE {state_names[i]}:  {mse_states_ukf[i]:.6f}")
        print(f"PF  MSE {state_names[i]}:  {mse_states_pf[i]:.6f}")
        print("-" * 40)

    # Individual plots for each state
    state_plots_info = [
        {'idx': 0, 'var': 'θ₁', 'desc': 'Angle of First Pendulum', 'y_label': 'Angle (rad)'},
        {'idx': 1, 'var': 'θ₂', 'desc': 'Angle of Second Pendulum', 'y_label': 'Angle (rad)'},
        {'idx': 2, 'var': 'ω₁', 'desc': 'Angular Velocity of First Pendulum', 'y_label': 'Angular Velocity (rad/s)'},
        {'idx': 3, 'var': 'ω₂', 'desc': 'Angular Velocity of Second Pendulum', 'y_label': 'Angular Velocity (rad/s)'}
    ]

    for state_info in state_plots_info:
        i = state_info['idx']

        fig = go.Figure()

        # True state (black thick line)
        fig.add_trace(go.Scatter(
            x=t_history, y=x_true[:, i],
            mode='lines',
            name='True State',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))

        # Noisy measurements (purple dots)
        # Noisy measurements are already in x_true (which includes measurement noise)
        # So we plot x_true as the measurements
        fig.add_trace(go.Scatter(
            x=t_history, y=x_true[:, i],
            mode='markers',
            name='Noisy Measurements',
            marker=dict(size=2.5, color='rgba(138, 43, 226, 0.6)'),
            showlegend=True
        ))

        # UKF estimation
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))

        # PF estimation
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))

        # Set legend position
        legend_y = 0.98
        legend_yanchor = 'top'

        fig.update_layout(
            title={
                'text': f'State {state_info["var"]}: {state_info["desc"]} - Double Pendulum',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Time (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.98,
                y=legend_y,
                xanchor='right',
                yanchor=legend_yanchor,
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )

        fig.show()

    # Combined error plot
    fig_err = go.Figure()

    colors = ['#2ca02c', '#d62728', '#9467bd', '#8c564b']  # Different colors for each state

    for i in range(num_neurons):
        error_ukf = x_true[:, i] - x_hat_ukf[:, i]
        error_pf = x_true[:, i] - x_hat_pf[:, i]

        # UKF errors
        fig_err.add_trace(go.Scatter(
            x=t_history, y=error_ukf,
            mode='lines',
            name=f'UKF Error {state_names[i]} (MSE={mse_states_ukf[i]:.2e})',
            line=dict(color=colors[i], width=1.5),
            opacity=0.7
        ))

        # PF errors (dashed)
        fig_err.add_trace(go.Scatter(
            x=t_history, y=error_pf,
            mode='lines',
            name=f'PF Error {state_names[i]} (MSE={mse_states_pf[i]:.2e})',
            line=dict(color=colors[i], width=1.5, dash='dash'),
            opacity=0.7
        ))

    fig_err.update_layout(
        title={
            'text': 'Estimation Errors - Double Pendulum',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Time (s)',
        yaxis_title='Estimation Error',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_err.show()

    # Phase space plots
    # θ1 vs ω1
    fig_phase1 = go.Figure()

    fig_phase1.add_trace(go.Scatter(
        x=x_true[:, 0], y=x_true[:, 2],
        mode='lines',
        name='True Phase Space (θ₁ vs ω₁)',
        line=dict(color='#000000', width=2)
    ))

    fig_phase1.add_trace(go.Scatter(
        x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 2],
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=1.5, dash='dot')
    ))

    fig_phase1.add_trace(go.Scatter(
        x=x_hat_pf[:, 0], y=x_hat_pf[:, 2],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=1.5, dash='dashdot')
    ))

    fig_phase1.update_layout(
        title={
            'text': 'Phase Space: θ₁ vs ω₁ - Double Pendulum',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='θ₁ (rad)',
        yaxis_title='ω₁ (rad/s)',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.02,
            xanchor='right',
            yanchor='bottom',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_width'],
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_phase1.show()

    # Bar chart comparing MSE
    fig_mse = go.Figure()

    filters = ['UKF-RHONN', 'PF-RHONN']

    # Add bars for each state
    for i in range(num_neurons):
        fig_mse.add_trace(go.Bar(
            name=state_names[i],
            x=filters,
            y=[mse_states_ukf[i], mse_states_pf[i]],
            text=[f'{mse_states_ukf[i]:.2e}', f'{mse_states_pf[i]:.2e}'],
            textposition='outside'
        ))

    fig_mse.update_layout(
        title={
            'text': 'Mean Squared Error (MSE) Comparison - Double Pendulum',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Filter Type',
        yaxis_title='Mean Squared Error (MSE)',
        yaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_mse.show()

    # ============================================================
    # 7) RMSE Calculation and Plots
    # ============================================================

    # Calculate RMSE for each filter and state
    rmse_states_ukf = [np.sqrt(mse) for mse in mse_states_ukf]
    rmse_states_pf = [np.sqrt(mse) for mse in mse_states_pf]

    # Total RMSE (combined metric)
    rmse_total_ukf = np.sqrt(mse_total_ukf)
    rmse_total_pf = np.sqrt(mse_total_pf)

    # Print RMSE comparison
    print("\n" + "="*70)
    print("📊 ROOT MEAN SQUARE ERROR (RMSE) COMPARISON")
    print("="*70)
    print("\nUKF-RHONN:")
    for i in range(num_neurons):
        print(f"  RMSE {state_names[i]}: {rmse_states_ukf[i]:.6f}")
    print(f"  RMSE Total: {rmse_total_ukf:.6f}")

    print("\nPF-RHONN:")
    for i in range(num_neurons):
        print(f"  RMSE {state_names[i]}: {rmse_states_pf[i]:.6f}")
    print(f"  RMSE Total: {rmse_total_pf:.6f}")

    print("\n" + "="*70)
    print(f"🏆 BEST FILTER (RMSE): ", end="")
    rmse_dict = {'UKF': rmse_total_ukf, 'PF': rmse_total_pf}
    best_filter_rmse = min(rmse_dict, key=rmse_dict.get)
    print(f"{best_filter_rmse} (Total RMSE: {rmse_dict[best_filter_rmse]:.6f})")
    print("="*70)

    print("\nSimulation completed successfully!")
    print(f"\nSummary:")
    print(f"  - System: Double Pendulum (4 states)")
    print(f"  - Simulation time: {t_history[-1]:.2f} seconds")
    print(f"  - Time step: {dt:.3f} seconds")
    print(f"  - Best filter: {best_filter} (Total RMSE: {rmse_dict[best_filter]:.6f})")

Double Pendulum System Identification
Number of features per neuron: [3, 3, 4, 4]
Total features: 14
States: [θ1, θ2, ω1, ω2]

Common Initial Weights (different dimensions per neuron):
  Neuron 0 (θ1 (angle 1)): shape = (3,)
  Neuron 1 (θ2 (angle 2)): shape = (3,)
  Neuron 2 (ω1 (velocity 1)): shape = (4,)
  Neuron 3 (ω2 (velocity 2)): shape = (4,)

Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 80.0%
Simulation progress: 90.0%
Simulation finished.

🏆 BEST FILTER: UKF (Total MSE: 436.775807)

Final Weights UKF-RHONN (different equations per neuron):
  Neuron 0 (θ1 (angle 1)): shape = (3,)
  Neuron 1 (θ2 (angle 2)): shape = (3,)
  Neuron 2 (ω1 (velocity 1)): shape = (4,)
  Neuron 3 (ω2 (velocity 2)): shape = (4,)

Weight Estimates PF-RHONN (different equations per neuron):
  Neu


📊 ROOT MEAN SQUARE ERROR (RMSE) COMPARISON

UKF-RHONN:
  RMSE θ1 (angle 1): 20.898915
  RMSE θ2 (angle 2): 0.027564
  RMSE ω1 (velocity 1): 0.050420
  RMSE ω2 (velocity 2): 0.088745
  RMSE Total: 20.899182

PF-RHONN:
  RMSE θ1 (angle 1): 20.932434
  RMSE θ2 (angle 2): 0.023534
  RMSE ω1 (velocity 1): 0.020139
  RMSE ω2 (velocity 2): 0.020152
  RMSE Total: 20.932466

🏆 BEST FILTER (RMSE): UKF (Total RMSE: 20.899182)

Simulation completed successfully!

Summary:
  - System: Double Pendulum (4 states)
  - Simulation time: 9.99 seconds
  - Time step: 0.010 seconds
  - Best filter: UKF (Total RMSE: 20.899182)
